## Example 1: Serialize a document to JSON bytes

In [ ]:
import importlib.metadata
import json

from smithy_core.documents import Document
from smithy_json import JSONCodec

# Example 1: the codec writes compact JSON from a Smithy document
assert importlib.metadata.version("smithy-json") == "0.3.0"
codec = JSONCodec()
raw = codec.serialize(Document({"model": Document("nova"), "temperature": Document(0.5), "stop": Document([Document("\n")])}))
assert json.loads(raw) == {"model": "nova", "temperature": 0.5, "stop": ["\n"]}, raw
assert codec.media_type == "application/json"
print("Example 1 OK:", raw)

## Example 2: Read JSON back as a document

In [ ]:
from smithy_core import prelude
from smithy_json import JSONCodec

# Example 2: a deserializer streams the bytes (through ijson) into a document
codec = JSONCodec()
doc = codec.create_deserializer(b'{"messages": [{"role": "user", "content": "hi"}], "n": 2}').read_document(prelude.DOCUMENT)
assert doc.as_value() == {"messages": [{"role": "user", "content": "hi"}], "n": 2}, doc.as_value()
print("Example 2 OK:", doc.as_value())

## Example 3: Read typed scalars

In [ ]:
from smithy_core import prelude
from smithy_json import JSONCodec

# Example 3: each read_* method checks the JSON type against the schema
codec = JSONCodec()
assert codec.create_deserializer(b"42").read_integer(prelude.INTEGER) == 42
assert codec.create_deserializer(b'"text"').read_string(prelude.STRING) == "text"
assert codec.create_deserializer(b"true").read_boolean(prelude.BOOLEAN) is True
assert codec.create_deserializer(b"null").is_null()
print("Example 3 OK")

## Example 4: Blobs are base64 on the wire

In [ ]:
import base64

from smithy_core import prelude
from smithy_json import JSONCodec

# Example 4: binary data travels as base64 text and comes back as bytes
codec = JSONCodec()
raw = base64.b64encode(b"\x00\x01binary").decode()
blob = codec.create_deserializer(f'"{raw}"'.encode()).read_blob(prelude.BLOB)
assert blob == b"\x00\x01binary", blob
print("Example 4 OK:", raw)

## Example 5: Large documents round-trip

In [ ]:
from smithy_core import prelude
from smithy_core.documents import Document
from smithy_json import JSONCodec

# Example 5: a document with 1,000 entries survives serialize -> deserialize unchanged
codec = JSONCodec()
original = {f"key{i}": i * 0.5 for i in range(1000)}
raw = codec.serialize(Document({k: Document(v) for k, v in original.items()}))
back = codec.create_deserializer(raw).read_document(prelude.DOCUMENT).as_value()
assert back == original and len(raw) > 10000
print("Example 5 OK:", len(raw), "bytes")